<a href="https://colab.research.google.com/github/umeshgurjar914042-creator/capstone-faq-bot/blob/main/capstone_retrieval_day2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [10]:
!pip install -q sentence-transformers chromadb

### Embedding and Querying with Sentence-Transformers and ChromaDB

In this section, we will initialize a local ChromaDB database, define a helper class or method to generate embeddings using `sentence-transformers`, and then store and query some sample documents.

In [11]:
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize the sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Initialize ChromaDB client (in-memory by default)
chroma_client = chromadb.Client()

# Create a new collection
collection = chroma_client.get_or_create_collection(name="my_collection")

# Define documents and metadata
documents = [
    "ChromaDB is an open-source vector database designed for AI applications.",
    "Sentence-transformers allows you to generate dense vector representations of text.",
    "Python is a popular programming language widely used in data science and machine learning."
]
ids = ["doc1", "doc2", "doc3"]

# Generate embeddings
embeddings = model.encode(documents).tolist()

# Add items to the collection
collection.add(
    embeddings=embeddings,
    documents=documents,
    ids=ids
)

print("Successfully added documents and their embeddings to ChromaDB!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Successfully added documents and their embeddings to ChromaDB!


In [12]:
# Query the database with a new sentence
query_text = "How do I embed text data?"
query_embedding = model.encode([query_text]).tolist()

results = collection.query(
    query_embeddings=query_embedding,
    n_results=1
)

print("Query:", query_text)
print("Most similar document:", results['documents'][0][0])
print("Distance:", results['distances'][0][0])

Query: How do I embed text data?
Most similar document: Sentence-transformers allows you to generate dense vector representations of text.
Distance: 1.0727543830871582


In [13]:
from sentence_transformers import SentenceTransformer
import chromadb

model = SentenceTransformer("paraphrase-multilingual-MiniLM-L12-v2")

faq = [
    "बैच का समय सुबह 7 से 9 और शाम 5 से 7 बजे है।",
    "मासिक फ़ीस 1500 रुपये है।",
    "रविवार को कक्षा बंद रहती है।",
    "पहले दिन की डेमो कक्षा मुफ्त है।",
    "फ़ीस हर महीने की 10 तारीख तक जमा करनी होती है।",
    "एक बैच में अधिकतम 25 बच्चे होते हैं।",
]

emb = model.encode(faq, normalize_embeddings=True).tolist()
db = chromadb.Client()
col = db.get_or_create_collection("faq", metadata={"hnsw:space": "cosine"})
col.upsert(ids=[f"c{i}" for i in range(len(faq))], documents=faq, embeddings=emb)

def retrieve(q, k=3):
    e = model.encode([q], normalize_embeddings=True).tolist()
    r = col.query(query_embeddings=e, n_results=k)
    ids = [int(i[1:]) for i in r["ids"][0]]
    return ids, r["distances"][0]

print(retrieve("फ़ीस कितनी है?", k=1))

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

([1], [0.6226470470428467])


✅ फ़ीस कितनी है? -> [1]
❌ कक्षा कब लगती है? -> [2]
✅ संडे को क्लास होती है क्या? -> [2]
✅ पहली क्लास के पैसे लगेंगे? -> [3]
❌ पैसे कब तक देने हैं? -> [1]
✅ एक क्लास में कितने बच्चे पढ़ते हैं? -> [5]
k=1  hit-rate: 4/6  MRR: 0.67
✅ फ़ीस कितनी है? -> [1, 5, 0]
✅ कक्षा कब लगती है? -> [2, 3, 0]
✅ संडे को क्लास होती है क्या? -> [2, 3, 0]
✅ पहली क्लास के पैसे लगेंगे? -> [3, 5, 0]
✅ पैसे कब तक देने हैं? -> [1, 4, 0]
✅ एक क्लास में कितने बच्चे पढ़ते हैं? -> [5, 2, 4]
k=3  hit-rate: 6/6  MRR: 0.81


फ़ीस कितनी है? -> [1] 0.623
टीचर की सैलरी कितनी है? -> [5] 0.575
बारिश कब होगी? -> [0] 0.623
